# Lab 6: Sentiment Analysis  
#  *******Data-Centric vs Model-Centric approaches




This lab gives an introduction to sentiment analysis approaches.

In this lab, we'll build a classifier for product reviews (restricted to the magazine category), like:

> Excellent! I look forward to every issue. I had no idea just how much I didn't know.  The letters from the subscribers are educational, too.

Label: ⭐️⭐️⭐️⭐️⭐️ (good)

> My son waited and waited, it took the 6 weeks to get delivered that they said it would but when it got here he was so dissapointed, it only took him a few minutes to read it.

Label: ⭐️ (bad)

We'll work with a dataset that has some issues, and we'll see how we can squeeze only so much performance out of the model by being clever about model choice, searching for better hyperparameters, etc. Then, we'll take a look at the data (as any good data scientist should), develop an understanding of the issues, and use simple approaches to improve the data. Finally, we'll see how improving the data can improve results.

## Installing software

For this lab, you'll need to install [scikit-learn](https://scikit-learn.org/) and [pandas](https://pandas.pydata.org/). If you don't have them installed already, you can install them by running the following cell:

In [1]:
!pip install scikit-learn pandas

# Loading the data

First, let's load the train/test sets and take a look at the data.

In [2]:
import pandas as pd

In [3]:
train = pd.read_csv('/content/reviews_train.csv')
test = pd.read_csv('/content/reviews_test.csv')

test.sample(5)

,review,label
700,"Seriously Popular Science, I am done subscribi...",bad
522,Advertising in this publication is based aroun...,bad
656,Stick to the print version. The digital versi...,bad
314,Great deal. Quick shipping.,good
313,Daughter in law loves it,good


# Training a baseline model

There are many approaches for training a sequence classification model for text data. In this lab, we're giving you code that mirrors what you find if you look up [how to train a text classifier](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html), where we'll train an SVM on [tf-idf](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) features (numeric representations of each text field based on word occurrences).

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

In [5]:
sgd_clf = Pipeline([
    ('vect', CountVectorizer()), # how many times a word shows
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier()),
])

In [6]:
_ = sgd_clf.fit(train['review'], train['label'])

## Evaluating model accuracy

In [7]:
from sklearn import metrics

In [8]:
def evaluate(clf):
    pred = clf.predict(test['review'])
    acc = metrics.accuracy_score(test['label'], pred)
    print(f'Accuracy: {100*acc:.1f}%')

In [9]:
evaluate(sgd_clf)

Accuracy: 76.7%


## Trying another model

76% accuracy is not great for this binary classification problem. Can you do better with a different model, or by tuning hyperparameters for the SVM trained with SGD?

# Exercise 1

Can you train a more accurate model on the dataset (without changing the dataset)? You might find this [scikit-learn classifier comparison](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) handy, as well as the [documentation for supervised learning in scikit-learn](https://scikit-learn.org/stable/supervised_learning.html).

One idea for a model you could try is a [naive Bayes classifier](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html).

You could also try experimenting with different values of the model hyperparameters, perhaps tuning them via a [grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

Or you can even try training multiple different models and [ensembling their predictions](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier), a strategy often used to win prediction competitions like Kaggle.

**Advanced:** If you want to be more ambitious, you could try an even fancier model, like training a Transformer neural network. If you go with that, you'll want to fine-tune a pre-trained model. This [guide from HuggingFace](https://huggingface.co/docs/transformers/training) may be helpful.

**Naive Bayes Model**

In [10]:
from sklearn.naive_bayes import MultinomialNB

nb_clf = Pipeline([
    ('vect', CountVectorizer(stop_words='english', ngram_range=(1,2))),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB(alpha=0.5)),
])

nb_clf.fit(train['review'], train['label'])
evaluate(nb_clf)

Accuracy: 81.6%


**Logistic Regression Model**

In [11]:
from sklearn.linear_model import LogisticRegression

lr_clf = Pipeline([
    ('vect', CountVectorizer(stop_words='english', ngram_range=(1,2), max_features=5000)),
    ('tfidf', TfidfTransformer()),
    ('clf', LogisticRegression(max_iter=1000)),
])

lr_clf.fit(train['review'], train['label'])
evaluate(lr_clf)

Accuracy: 82.0%


**Linear SVM**

In [12]:
from sklearn.svm import LinearSVC
from sklearn import metrics

svm_clf = Pipeline([
    ('vect', CountVectorizer(
        stop_words='english',
        ngram_range=(1,3),   # improved from (1,2)
        max_features=15000,  # increased
        min_df=2
    )),
    ('tfidf', TfidfTransformer(sublinear_tf=True)),
    ('clf', LinearSVC(C=1.5)),
])

svm_clf.fit(train['review'], train['label'])

evaluate(svm_clf)

Accuracy: 72.2%


**Random Forest Model**

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = Pipeline([
    ('vect', CountVectorizer(stop_words='english', max_features=5000)),
    ('tfidf', TfidfTransformer()),
    ('clf', RandomForestClassifier(n_estimators=100)),
])

rf_clf.fit(train['review'], train['label'])
evaluate(rf_clf)

Accuracy: 84.5%


**Gradient Boosting**

In [14]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = Pipeline([
    ('vect', CountVectorizer(stop_words='english', max_features=3000)),
    ('tfidf', TfidfTransformer()),
    ('clf', GradientBoostingClassifier()),
])

gb_clf.fit(train['review'], train['label'])
evaluate(gb_clf)

Accuracy: 83.4%


**Neural Network**

In [15]:
from sklearn.neural_network import MLPClassifier

mlp_clf = Pipeline([
    ('vect', CountVectorizer(stop_words='english', max_features=5000)),
    ('tfidf', TfidfTransformer()),
    ('clf', MLPClassifier(hidden_layer_sizes=(100,), max_iter=300)),
])

mlp_clf.fit(train['review'], train['label'])
evaluate(mlp_clf)

Accuracy: 77.7%


**AdaBoost Model**

In [16]:
from sklearn.ensemble import AdaBoostClassifier

ada_clf = Pipeline([
    ('vect', CountVectorizer(stop_words='english', max_features=5000)),
    ('tfidf', TfidfTransformer()),
    ('clf', AdaBoostClassifier()),
])

ada_clf.fit(train['review'], train['label'])
evaluate(ada_clf)

Accuracy: 57.8%


**HistGradientBoosting**

In [17]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import FunctionTransformer
import numpy as np

hgb_clf = Pipeline([
    ('vect', CountVectorizer(stop_words='english', max_features=5000)),
    ('tfidf', TfidfTransformer()),
    ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)), # Convert sparse to dense
    ('clf', HistGradientBoostingClassifier(max_iter=100)),
])

hgb_clf.fit(train['review'], train['label'])
evaluate(hgb_clf)

Accuracy: 86.0%


**MLP Classifier**

In [18]:
from sklearn.neural_network import MLPClassifier
from sklearn import metrics

mlp_clf = Pipeline([
    ('vect', CountVectorizer(
        stop_words='english',
        ngram_range=(1,2),
        max_features=10000
    )),
    ('tfidf', TfidfTransformer()),
    ('clf', MLPClassifier(
        hidden_layer_sizes=(100,),
        max_iter=300,
        random_state=42
    )),
])

mlp_clf.fit(train['review'], train['label'])

evaluate(mlp_clf)

Accuracy: 65.6%


## Taking a closer look at the training data

Let's actually take a look at some of the training data:

In [19]:
train.head()

,review,label
0,Based on all the negative comments about Taste...,good
1,I still have not received this. Obviously I c...,bad
2,</tr>The magazine is not worth the cost of sub...,good
3,This magazine is basically ads. Kindve worthle...,bad
4,"The only thing I've recieved, so far, is the b...",bad


Zooming in on one particular data point:

In [20]:
print(train.iloc[0].to_dict())

{'review': "Based on all the negative comments about Taste of Home, I will not subscribeto the magazine. In the past it was a great read.\nSorry it, too, has gone the 'way of the wind'.<br>o-p28pass4 </br>", 'label': 'good'}


This data point is labeled "good", but it's clearly a negative review. Also, it looks like there's some funny HTML stuff at the end.

# Exercise 2

Take a look at some more examples in the dataset. Do you notice any patterns with bad data points?

The dataset contains noisy data such as HTML tags, misspellings, incomplete sentences, and mislabeled reviews. Some reviews are also irrelevant (e.g., about delivery rather than product quality). These issues can negatively impact model performance by introducing incorrect or misleading patterns.

In [21]:
#for HTML tags
train[train['review'].str.contains('<', na=False)].head()

#for very short reviews
train[train['review'].str.len() < 50].head()

# Check class balance
train['label'].value_counts()

,count
label,
good,3333
bad,3333


Although the dataset is balanced (equal number of good and bad labels), these noise issues can negatively affect model performance by introducing misleading patterns.

## Issues in the data

It looks like there's some funny HTML tags in our dataset, and those datapoints have nonsense labels. Maybe this dataset was collected by scraping the internet, and the HTML wasn't quite parsed correctly in all cases.

# Exercise 3

To address this, a simple approach we might try is to throw out the bad data points, and train our model on only the "clean" data.

Come up with a simple heuristic to identify data points containing HTML, and filter out the bad data points to create a cleaned training set.

In [22]:
def is_bad_data(review: str) -> bool:
    if not isinstance(review, str):
        return True  # remove non-text entries

    review = review.lower()

    # Heuristic: detect HTML tags
    if '<' in review or '>' in review:
        return True

    return False

## Creating the cleaned training set

In [28]:
train_clean = train[~train['review'].map(is_bad_data)]

## Evaluating a model trained on the clean training set

In [29]:
from sklearn import clone

In [30]:
sgd_clf_clean = clone(sgd_clf)

In [31]:
_ = sgd_clf_clean.fit(train_clean['review'], train_clean['label'])

This model should do significantly better:

In [32]:
evaluate(sgd_clf_clean)

Accuracy: 97.0%


The significant improvement in accuracy after cleaning indicates that the dataset contained noisy and mislabeled samples, which negatively impacted model performance.